In [1]:
import pandas as pd
import numpy as np
import random
import json

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel 
from transformers import logging

In [2]:
behaviors = pd.read_csv('train/train_behaviors.tsv', delimiter='\t', index_col=0, header=None)
behaviors.columns = ['user', 'time', 'clicked_news', 'impressions']
behaviors = behaviors.sample(frac=1, random_state=42)
train_behaviors = behaviors[:int(len(behaviors) * 0.8)]
valid_behaviors = behaviors[int(len(behaviors) * 0.8):]

news = pd.read_csv('train/train_news.tsv', delimiter='\t', header=None)
news.columns = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'URL', 'title_entities', 'abstract_entities']
news_dict = {data['news_id']: data.iloc[1:] for _, data in news.iterrows()}

embedding = {}
f = open("train/train_entity_embedding.vec")
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embedding[word] = coefs
f.close()

logging.set_verbosity_error()
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [8]:
class RecommendationDataset(Dataset):
    def __init__(self, behaviors, news_dict, embedding, tokenizer):
        self.behaviors = behaviors
        self.news_dict = news_dict
        self.embedding = embedding
        self.tokenizer = tokenizer
        
    def __len__(self):
        return len(self.behaviors)

    def padding(self, item, size):
        item = torch.stack(item)[:size]
        if item.size(0) < size:
            padding_length = size - item.size(0)
            padding_item = torch.zeros((padding_length, *item.shape[1:]))
            item = torch.cat((item, padding_item), dim=0)
        return item
    
    def extract_entities(self, news):
        _, _, _, _, _, title_entities, abstract_entities = self.news_dict[news]
        
        title_entities = "[]" if isinstance(title_entities, float) else title_entities
        abstract_entities = "[]" if isinstance(abstract_entities, float) else abstract_entities
        
        vector = []
        entities = json.loads(title_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.embedding[id_]) for id_ in ids if id_ in self.embedding]
        entities = json.loads(abstract_entities)
        ids = [entity['WikidataId'] for entity in entities]
        vector += [torch.tensor(self.embedding[id_]) for id_ in ids if id_ in self.embedding]

        if len(vector) == 0:
            vector = torch.zeros((30, 100))
        else:
            vector = self.padding(vector, 30)
                
        return vector

    def extract_text(self, news):
        category, subcategory, title, abstract, _, _, _ = self.news_dict[news]
        
        title = "" if isinstance(title, float) else title
        abstract = "" if isinstance(abstract, float) else abstract
        encoding = self.tokenizer(
            text = category + " " + subcategory,
            text_pair = title + " " + abstract,
            return_tensors = "pt",
            padding = "max_length",
            truncation = True,
            max_length = 200
        )
        return encoding
    
    def __getitem__(self, index):
        _, _, clicked_news, impressions = self.behaviors.iloc[index]
        
        history_vectors = []
        history_encodings = []
        clicked_news = clicked_news.split()
        for news in clicked_news:
            vector = self.extract_entities(news)
            encoding = self.extract_text(news)
            history_vectors.append(vector)
            history_encodings.append(encoding)
        history_vectors = self.padding(history_vectors, 20)
        history_ids = [encoding.input_ids[0, :] for encoding in history_encodings]
        history_ids = self.padding(history_ids, 20)
        history_type = [encoding.token_type_ids[0, :] for encoding in history_encodings]
        history_type = self.padding(history_type, 20)
        history_mask = [encoding.attention_mask[0, :] for encoding in history_encodings]
        history_mask = self.padding(history_mask, 20)

        recommen_vectors = []
        recommen_encodings = []
        impressions = impressions.split()
        labels = [int(impression.split('-')[1]) for impression in impressions]
        impression_news = [impression.split('-')[0] for impression in impressions]
        for news in impression_news:
            vector = self.extract_entities(news)
            encoding = self.extract_text(news)
            recommen_vectors.append(vector)
            recommen_encodings.append(encoding)
        recommen_ids = [encoding.input_ids for encoding in recommen_encodings]
        recommen_ids = torch.cat(recommen_ids)
        recommen_type = [encoding.token_type_ids for encoding in recommen_encodings]
        recommen_type = torch.cat(recommen_type)
        recommen_mask = [encoding.attention_mask for encoding in recommen_encodings]
        recommen_mask = torch.cat(recommen_mask)

        
        return history_vectors, history_ids, history_type, history_mask, \
            recommen_vectors, recommen_ids, recommen_type, recommen_mask, labels
    
train_dataset = RecommendationDataset(train_behaviors, news_dict, embedding, tokenizer)
valid_dataset = RecommendationDataset(valid_behaviors, news_dict, embedding, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)

In [6]:
class RecommendationModel(nn.Module):
    def __init__(self):
        super(RecommendationModel, self).__init__()
        
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        #self.entity_attention = 
        #self.merge_attention = 
    
    def forward(self, history_vectors, history_ids, history_type, history_mask, \
        recommen_vectors, recommen_ids, recommen_type, recommen_mask):
        pass
        

In [ ]:

for history_vectors, history_ids, history_type, history_mask, \
    recommen_vectors, recommen_ids, recommen_type, recommen_mask, labels in train_loader:
    print(history_vectors.shape, history_ids.shape, recommen_ids.shape)


torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 

torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 

torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 

torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 

torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 

...

torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 

torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 

torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 

torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 

torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 20, 30, 100]) torch.Size([16, 20, 200]) torch.Size([16, 15, 200])
torch.Size([16, 